# 2.1 Paragraph classification v2 — honest evaluation

Copy of `2.0_paragraph_classif.ipynb`. `model/`, `model_v2/`, `errors.csv` and
`error_summary.txt` are frozen; this notebook writes to `model_v3/` and `eval_v2/`.

Two things change, both about measurement rather than modelling:

1. **The same model is trained twice**, once per split column from `1.3`. `split` is the v1
   document-grouped split; `split_court` holds out whole courts. Hyperparameters, encoder and
   seed are identical, so the gap between the two test scores is the part of v1's score that
   came from having seen the court's template.
2. **Confidence intervals resample documents, not paragraphs.** Paragraph labels are strongly
   dependent within a judgment (P(next paragraph shares the section) = 0.909 against 0.366
   under independence), so a per-row interval is far too narrow. Both are reported.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict

from sklearn.metrics import accuracy_score, classification_report, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

In [ ]:
DATASET_PATH = "legal_paragraphs_dataset_v2.parquet"
ENCODER_PATH = "bge_model/bge-small-uk-law-v1_1/final/encoder"
TOKENIZER_PATH = "bge_model/bge-small-uk-law-v1_1/final/tokenizer"

MODEL_ROOT = Path("model_v3")
EVAL_ROOT = Path("eval_v2")
EVAL_ROOT.mkdir(exist_ok=True)

SPLIT_COLUMNS = ["split", "split_court"]
SEED = 42
N_BOOTSTRAP = 1000
QUALITY_FLAGS = ["section_scrambled", "section_missing", "extreme_length"]

In [ ]:
p_df = pd.read_parquet(DATASET_PATH)
p_df["document_id"] = p_df["document_id"].astype(str)
p_df.shape

In [ ]:
LABELS_TO_IDS = {"introductory": 0, "reasoning": 1, "operative": 2}
IDS_TO_LABELS = {value: key for key, value in LABELS_TO_IDS.items()}
assert p_df.drop_duplicates(["section", "label"]).set_index("section")["label"].to_dict() == LABELS_TO_IDS

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["context"],
        batch["text"],
        max_length=512,
        padding=False,
    )

def compute_metrics(prediction):
    logits, labels = prediction
    predicted_labels = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, predicted_labels),
        "macro_f1": f1_score(labels, predicted_labels, average="macro", zero_division=0),
    }

## Train once per split column

In [ ]:
def build_datasets(split_column):
    columns = ["context", "text", "label"]
    dataset = DatasetDict({
        split: Dataset.from_pandas(
            p_df.loc[p_df[split_column] == split, columns].reset_index(drop=True),
            preserve_index=False,
        )
        for split in ["train", "validation", "test"]
    })
    return dataset.map(tokenize, batched=True, remove_columns=["context", "text"])

def train_for_split(split_column):
    tokenized = build_datasets(split_column)

    model = AutoModelForSequenceClassification.from_pretrained(
        ENCODER_PATH,
        num_labels=3,
        id2label=IDS_TO_LABELS,
        label2id=LABELS_TO_IDS,
    )

    args = TrainingArguments(
        output_dir=str(MODEL_ROOT / split_column),
        learning_rate=1e-5,
        weight_decay=0.01,
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=2,
        logging_steps=25,
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainer.train()

    logits = trainer.predict(tokenized["test"]).predictions
    test_rows = p_df.loc[p_df[split_column] == "test"].reset_index(drop=True)
    predictions = test_rows[
        ["document_id", "paragraph_index", "section", "label", "court", *QUALITY_FLAGS]
    ].copy()
    predictions["predicted"] = np.argmax(logits, axis=1)
    predictions["correct"] = predictions["predicted"] == predictions["label"]

    trainer.save_model(str(MODEL_ROOT / split_column / "final"))
    predictions.to_parquet(EVAL_ROOT / f"predictions_{split_column}.parquet", index=False)
    return predictions

In [ ]:
predictions = {}
for split_column in SPLIT_COLUMNS:
    print(f"\n{'=' * 70}\ntraining on {split_column}\n{'=' * 70}")
    predictions[split_column] = train_for_split(split_column)

## Cluster bootstrap

Resampling paragraphs independently would treat 28k rows as 28k pieces of evidence. They are
not: they come from ~850 documents, and within a document the labels run in long blocks. The
cluster bootstrap resamples whole documents, which is the level at which the corpus is
(approximately) exchangeable.

In [ ]:
def macro_f1_and_accuracy(labels, predicted):
    return (
        f1_score(labels, predicted, average="macro", zero_division=0),
        accuracy_score(labels, predicted),
    )

def bootstrap_interval(frame, cluster_column=None, n_bootstrap=N_BOOTSTRAP, seed=SEED):
    """Percentile CIs. Resamples clusters if given a column, else independent rows."""
    rng = np.random.default_rng(seed)
    labels = frame["label"].to_numpy()
    predicted = frame["predicted"].to_numpy()

    if cluster_column is None:
        draws = [
            rng.integers(0, len(frame), len(frame))
            for _ in range(n_bootstrap)
        ]
    else:
        codes = pd.factorize(frame[cluster_column])[0]
        by_cluster = [np.flatnonzero(codes == code) for code in range(codes.max() + 1)]
        draws = [
            np.concatenate([by_cluster[i] for i in rng.integers(0, len(by_cluster), len(by_cluster))])
            for _ in range(n_bootstrap)
        ]

    scores = np.array([macro_f1_and_accuracy(labels[idx], predicted[idx]) for idx in draws])
    point = macro_f1_and_accuracy(labels, predicted)
    low, high = np.percentile(scores, [2.5, 97.5], axis=0)
    return {
        "macro_f1": point[0], "macro_f1_lo": low[0], "macro_f1_hi": high[0],
        "accuracy": point[1], "accuracy_lo": low[1], "accuracy_hi": high[1],
    }

In [ ]:
intervals = []
for split_column, frame in predictions.items():
    for name, cluster in [("cluster (by document)", "document_id"), ("naive (by paragraph)", None)]:
        row = {"split": split_column, "bootstrap": name, **bootstrap_interval(frame, cluster)}
        row["macro_f1_width"] = row["macro_f1_hi"] - row["macro_f1_lo"]
        intervals.append(row)

intervals_df = pd.DataFrame(intervals)
intervals_df.round(4)

## Breakdown by court exposure and by quality flag

In [ ]:
def court_exposure(split_column):
    """How many training documents share each test document's court."""
    documents = p_df.drop_duplicates("document_id")
    train_counts = documents.loc[documents[split_column] == "train", "court"].value_counts()

    def bucket(court):
        if pd.isna(court):
            return "unattributed"
        seen = int(train_counts.get(court, 0))
        if seen == 0:
            return "unseen court"
        return "court seen 1-4x" if seen < 5 else "court seen 5x+"

    return documents.set_index("document_id")["court"].map(bucket)

exposure_tables = {}
for split_column, frame in predictions.items():
    frame = frame.copy()
    frame["exposure"] = frame["document_id"].map(court_exposure(split_column))
    table = pd.DataFrame([
        {
            "exposure": exposure,
            "n_documents": group["document_id"].nunique(),
            "n_paragraphs": len(group),
            "accuracy": group["correct"].mean(),
            "macro_f1": f1_score(group["label"], group["predicted"], average="macro", zero_division=0),
        }
        for exposure, group in frame.groupby("exposure")
    ])
    exposure_tables[split_column] = table
    print(f"--- {split_column} ---")
    print(table.round(4).to_string())
    print()

In [ ]:
flag_tables = {}
for split_column, frame in predictions.items():
    rows = []
    for flag in QUALITY_FLAGS:
        for value in [False, True]:
            subset = frame[frame[flag] == value]
            if subset.empty:
                continue
            rows.append({
                "flag": flag,
                "value": value,
                "n_documents": subset["document_id"].nunique(),
                "n_paragraphs": len(subset),
                "accuracy": subset["correct"].mean(),
                "macro_f1": f1_score(subset["label"], subset["predicted"], average="macro", zero_division=0),
            })
    flag_tables[split_column] = pd.DataFrame(rows)
    print(f"--- {split_column} ---")
    print(flag_tables[split_column].round(4).to_string(index=False))
    print()

In [ ]:
for split_column, frame in predictions.items():
    print(f"--- {split_column} ---")
    print(classification_report(
        frame["label"], frame["predicted"],
        target_names=[IDS_TO_LABELS[i] for i in range(3)],
        digits=4, zero_division=0,
    ))

## Write the summary

In [ ]:
def markdown_table(frame, floatfmt="{:.4f}"):
    frame = frame.copy()
    for column in frame.columns:
        if pd.api.types.is_float_dtype(frame[column]):
            frame[column] = frame[column].map(floatfmt.format)
    header = "| " + " | ".join(str(c) for c in frame.columns) + " |"
    rule = "| " + " | ".join("---" for _ in frame.columns) + " |"
    body = ["| " + " | ".join(str(v) for v in row) + " |" for row in frame.astype(str).values]
    return "\n".join([header, rule, *body])

cluster_rows = intervals_df[intervals_df["bootstrap"] == "cluster (by document)"].set_index("split")
honest = cluster_rows.loc["split_court", "macro_f1"]
optimistic = cluster_rows.loc["split", "macro_f1"]

# Recomputed here rather than quoted from 1.3, so the summary cannot go stale.
documents = p_df.drop_duplicates("document_id")
train_courts = set(documents.loc[documents["split"] == "train", "court"].dropna())
v1_test = documents.loc[documents["split"] == "test"]
v1_attributed = v1_test.loc[v1_test["court"].notna()]
v1_leak = v1_attributed["court"].isin(train_courts).mean()

lines = [
    "# Honest evaluation of the paragraph section classifier",
    "",
    "Generated by `2.1_paragraph_classif_v2.ipynb` from `legal_paragraphs_dataset_v2.parquet`.",
    "Identical model, encoder, hyperparameters and seed in both arms; only the split differs.",
    "",
    f"- `split` — v1 grouping, by `document_id`. {v1_leak:.1%} of the {len(v1_attributed)} "
    f"court-attributed test documents (of {len(v1_test)} total) share a court with train.",
    "- `split_court` — grouping by extracted court. Zero train/test court overlap.",
    "",
    f"**Test macro-F1: {optimistic:.4f} (document-grouped) vs {honest:.4f} (court-grouped), "
    f"a gap of {optimistic - honest:+.4f}.**",
    "",
    "## Test metrics with 95% bootstrap intervals",
    "",
    markdown_table(intervals_df[[
        "split", "bootstrap", "macro_f1", "macro_f1_lo", "macro_f1_hi", "macro_f1_width",
        "accuracy", "accuracy_lo", "accuracy_hi",
    ]]),
    "",
    "The naive interval resamples paragraphs independently, which the data rejects",
    "(P(next paragraph shares the section) = 0.909 vs 0.366 under independence). Compare the",
    "`macro_f1_width` column: the naive interval is the one to distrust.",
    "",
    "## By court exposure",
    "",
]
for split_column, table in exposure_tables.items():
    lines += [f"### `{split_column}`", "", markdown_table(table), ""]

lines += ["## By data-quality flag (from `1.1_data_cleaning.ipynb`)", ""]
for split_column, table in flag_tables.items():
    lines += [f"### `{split_column}`", "", markdown_table(table), ""]

(EVAL_ROOT / "summary.md").write_text("\n".join(lines), encoding="utf-8")
intervals_df.to_csv(EVAL_ROOT / "intervals.csv", index=False)
print("\n".join(lines))